In [ ]:
import handcalcs.render
from handcalcs import render
import handcalcs
from math import pi, sqrt
from handcalcs.decorator import handcalc
import forallpeople as si
from structuralcodes import set_design_code
from structuralcodes.materials.concrete import create_concrete
from math import sqrt, pi
from math import tan, radians, pi, exp
# 1. LOAD UNITS
# 'top_level=True' is the magic key. 
# It lets you type 'mm' instead of 'si.mm'
si.environment('structural', top_level=True)

# 2. LOAD DESIGN CODES
set_design_code('ec2_2004')


handcalcs.set_option("display_precision", 3)
handcalcs.set_option("underscore_subscripts", True)

# Custom symbol mappings (Python name -> LaTeX)
handcalcs.set_option("custom_symbols", {
    "Lambda_": r"\Lambda",   # uppercase Lambda (Λ)
    "lambda_": r"\lambda",  # lowercase lambda (λ) when using 'lambda_' in Python
    "V_dot": r"\dot{V}",
    "N_star": r"N^{*}",
})
print("Custom symbols registered.")

from sections_db import SectionDB, default_db_path

db = SectionDB(default_db_path()).load()

prefer = ["BSShapes2006", "ArcelorMittal_British", "ArcelorMittal_Europe"]  # edit to your taste


Custom symbols registered.


# INPUTS

In [13]:
# %%render param  
# #FOR CARBON STEEL MATERIAL
# # costants
# E_MPa = 210000.0
# nu = 0.30
# G_MPa = E_MPa / (2*(1+nu))
# gamma_M0 = 1.0
# gamma_M1 = 1.0
# alpha_LT = 0.49
# C1 = 1.0
# h_mm =50
# t_mm = 12 
# L_b_mm = 1000 #laterally unrestrained length
# fy_MPa = 235


In [14]:
%%render param
# constants FOR STAINLESS STEEL SS304
E_MPa = 193000.0
nu = 0.30
G_MPa = E_MPa / (2*(1+nu))
gamma_M0 = 1.0
gamma_M1 = 1.0
alpha_LT = 0.49
C1 = 1.0

h_mm = 100
t_mm = 12

L_b_mm = 290   # laterally unrestrained length
fy_MPa = 205    # SS304 grade 


<IPython.core.display.Latex object>

In [15]:
%%render param
# Section props
W_pl_mm3 = t_mm * h_mm**2 / 4
I_z_mm4  = h_mm * t_mm**3 / 12

# Rectangle dims for torsion J (force b <= h without min/max)
b0 = t_mm
h0 = h_mm
swap = b0 > h0
b = b0*(1 - swap) + h0*swap
h = h0*(1 - swap) + b0*swap





<IPython.core.display.Latex object>

## J approximation

In [16]:
%%render param
# J approximation broken into pieces (handcalcs-friendly)
r = b / h
r4 = r**4
term1 = 1/3
term2 = 0.21 * r
term3 = 1 - r4/12
coeff = term1 - term2 * term3
J_mm4 = b * h**3 * coeff



<IPython.core.display.Latex object>

In [17]:
%%render long
# Elastic critical moment (Iw neglected, conservative)
inside = E_MPa * I_z_mm4 * G_MPa * J_mm4
root_inside = sqrt(inside)
M_cr_Nmm = C1 * (pi / L_b_mm) * root_inside


<IPython.core.display.Latex object>

## Slenderness and reduction



In [18]:
%%render long
num = W_pl_mm3 * fy_MPa
ratio = num / M_cr_Nmm
lambda__LT = sqrt(ratio)
phi = 0.5 * (1 + alpha_LT*(lambda__LT - 0.2) + lambda__LT**2)
rad = phi**2 - lambda__LT**2 # > 0 to ensure not negative inside sqrt
chi_LT_raw = 1 / (phi + (phi**2 - lambda__LT**2)**0.5)
chi_LT = min(chi_LT_raw, 1.0) # Reduction factor



<IPython.core.display.Latex object>

## Resistances

In [19]:
%%render long
M_pl_Rd_Nmm = W_pl_mm3 * fy_MPa / gamma_M0 *10**-6 #KNm 
M_b_Rd_Nmm  = chi_LT * W_pl_mm3 * fy_MPa / gamma_M1 *10**-6 #KNm

<IPython.core.display.Latex object>